In [32]:
import arxiv
from sentence_transformers import SentenceTransformer
import chromadb
import hashlib
from openai import OpenAI

In [58]:
groq_key = "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"
model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.EphemeralClient()
client.delete_collection("store")
store = client.create_collection("store")
arxClient = arxiv.Client(
    delay_seconds=3,
    num_retries=3
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4749.85it/s]


In [59]:
def ask(question):
    chat_client = OpenAI(
        api_key=groq_key,
        base_url="https://api.groq.com/openai/v1"
    )
    resp = chat_client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {"role": "user", "content": question}
        ]
    )
    return resp.choices[0].message.content


In [66]:
import time
def get_papers(query, max_results=5):
    url = 'https://export.arxiv.org/api/query'
    search = arxiv.Search(
        query=query,
        max_results=2,
        sort_by=arxiv.SortCriterion.Relevance
    )
    time.sleep(3)
    papers = []
    for paper in arxClient.results(search):
        title = paper.title
        sum = paper.summary
        authors = [i.name for i in paper.authors]
        category = paper.categories
        print(paper.links[0])
        print(paper.links[1])
        papers.append({"title": title, "summary": sum, "authors": authors,"category":category})
    return papers
temp = get_papers("testing of drug")


KeyboardInterrupt: 

In [37]:
def query_construct(question):
    prompt = f"""
    You are an AI research assistant. Given a question, generate the best possible query to search for the arxiv api for the question : {question}
    ti:	Title
    au:	Author
    abs:	Abstract
    co	:Comment
    jr	:Journal Reference
    cat	:Subject Category;
    Use Boolean logic, synonyms, field prefixes mentioned above, and relevant categories with `cat:`. Prefer precision over noise. Only output the search query, an example of the query for question anti-gravity is
    (ti:antigravity OR ti:"anti-gravity" OR abs:antigravity OR abs:"anti-gravity" OR all:antigravity OR all:"anti-gravity" OR all:"repulsive gravity" OR all:"negative mass") AND (cat:physics.gen-ph OR cat:gr-qc OR cat:physics.class-ph)
    and only output the query nothing else
    """
    resp = ask(prompt)
    return resp


In [38]:
def distill(text):
    return ask(f"""Distill the following research paper summary into 3-4 sentences capturing only the core idea, key method, and main finding. Return only the distilled summary, no preamble.

Summary: {text}""")

In [39]:

def indexing(topic, max_results=4):
    query = query_construct(topic)
    papers = get_papers(query, max_results)
    texts = [f"{p['title']} :- {p['summary']}" for p in papers]
    ids = [hashlib.md5(p['title'].encode()).hexdigest() for p in papers]
    existing_ids = store.get(ids=ids)['ids']
    new_titles = []
    new_ids = []
    new_text = []
    small_summaries = []
    for num, i in enumerate(ids):
        if i not in existing_ids:
            new_ids.append(i)
            new_text.append(texts[num])
            new_titles.append(papers[num]['title'])
            small_summaries.append(distill(texts[num]))
    embeddings = model.encode(small_summaries).tolist()
    if len(new_ids) > 0:
        store.add(
            documents=new_text,
            ids=new_ids,
            embeddings=embeddings,
            metadatas=[{"title":t} for t in new_titles]
        )
    print(f"Stored {len(new_text)} papers in vector store")

In [41]:
def retrival(query, n_res=3):
    embedded_ques = model.encode(query).tolist()
    res = store.query(query_embeddings=[embedded_ques], n_results=n_res)
    docs = res['documents'][0]
    titles = [m["title"] for m in res['metadatas'][0]]
    return list(zip(titles, docs))


In [40]:
def multi_query_generation(question, n=3):
    prompt = f"""You are an AI research assistant. Given a question, generate {n} different
    search queries to find relevant papers on arXiv.

    Each query should be 3-6 plain keywords, no quotes, no site: operators, no numbering.
    Return ONLY the queries, one per line.

    Question: {question}"""
    resp = ask(prompt)
    queries = resp.split('\n')
    queries = [q.strip() for q in queries if q.strip()]  # remove empty lines
    return queries

In [42]:
def reciprocal_rank_fusion(results_list, k=60):
    scores = {}

    for results in results_list:
        for rank, doc in enumerate(results):
            if doc not in scores:
                scores[doc] = 0
            # higher rank = lower position number = higher score
            scores[doc] += 1 / (rank + k)

    # sort by score, return just the texts
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in sorted_docs]


In [43]:
def grade(question,doc):
    prompt = f"""You are grading the relevance of a research paper to a question.

    Question: {question}
    Document: {doc}

    Grade the document:
    1 - Relevant and directly answers the question
    2 - Partially relevant, tangentially related
    3 - Completely irrelevant

    Return ONLY the number 1, 2, or 3. Nothing else."""
    return ask(prompt).strip()

In [44]:
def fusion_retrival(question, n=3):
    queries = multi_query_generation(question, n)
    all_context = []
    for query in queries:
        indexing(query)
        retrieved_docs = retrival(query)
        temp = []
        for title,doc in retrieved_docs:
            relevance = int(grade(question, doc))
            if relevance == 1:
                temp.append((title, doc))
            if relevance == 2:
                temp.append((title, distill(doc)))
        all_context.append(temp)
    ranked_list = reciprocal_rank_fusion(all_context)
    return ranked_list[:n]

In [45]:
def generation(question):
    context = "\n\n".join([f"[{title}]: {doc}" for title, doc in fusion_retrival(question)])
    prompt = f"""You are a research assistant answering questions based on provided papers.
    {context}
    Question: {question}
    Write a clear, concise answer in 5-6 sentences. Cite papers by their actual title in brackets like [Title].
    Do not repeat the same point twice. Write in plain English, not bullet points."""
    resp = ask(prompt)
    return resp

In [46]:
print(generation("how does machine learning help in drug discovery"))

KeyboardInterrupt: 